# 🎮 Démo RPG
Démonstration interactive de toutes les features du projet.

| Feature | Statut |
|---|---|
| Endurance / Niveau / Force | ✅ Prête |
| Dégâts aléatoires (0 → D) | ✅ Prête |
| Duel 2v2 | ✅ Prête |
| Armure selon équipement | ✅ Prête |
| Dégâts selon l'arme | ✅ Prête |

In [1]:
import sys
sys.path.insert(0, 'src')

from rpg.character import Character
from rpg.equipment import Equipment
from rpg.weapon import Weapon
from rpg.team import Team, duel

print('✅ Imports OK')

✅ Imports OK


## ⚔️ Feature 1 — Statistiques : Endurance, Niveau, Force
- `HP = 10 + endurance + 2 × niveau`
- `D max = (weapon.damage ou 1) + force + 2 × niveau`

In [2]:
def d_max(c):
    base = c.weapon.damage if c.weapon else 1
    return base + c.force + 2 * c.level

personnages = [
    Character('Alice'),
    Character('Tank',    endurance=5),
    Character('Vétéran', level=3),
    Character('Guerrier', force=4),
    Character('Héros',   endurance=3, level=2, force=2),
]

print(f'{"Personnage":<35} {"HP":>4}  {"D max":>6}')
print('-' * 50)
for c in personnages:
    label = f'{c.name} (END={c.endurance}, LVL={c.level}, FOR={c.force})'
    print(f'  {label:<33} {c.health:>4}  {d_max(c):>6}')

Personnage                            HP   D max
--------------------------------------------------
  Alice (END=0, LVL=0, FOR=0)         10       1
  Tank (END=5, LVL=0, FOR=0)          15       1
  Vétéran (END=0, LVL=3, FOR=0)       16       7
  Guerrier (END=0, LVL=0, FOR=4)      10       5
  Héros (END=3, LVL=2, FOR=2)         17       7


## 🎲 Feature 2 — Dégâts aléatoires entre 0 et D
À chaque attaque, les dégâts sont tirés aléatoirement entre `0` et `D max`.  
Les tests mockent `randint` pour garantir des résultats déterministes.

In [3]:
attacker = Character('Attaquant', force=3, level=1)
d = 1 + attacker.force + 2 * attacker.level
print(f'Attaquant (FOR=3, LVL=1) → D max = {d}')
print()
print('Simulation de 10 attaques :')
for i in range(10):
    target = Character('Cible')
    hp_before = target.health
    attacker.attack(target)
    degats = hp_before - target.health
    barre = '█' * degats
    print(f'  Attaque {i+1:2d} → dégâts: {degats:2d}  {barre}')

Attaquant (FOR=3, LVL=1) → D max = 6

Simulation de 10 attaques :
  Attaque  1 → dégâts:  2  ██
  Attaque  2 → dégâts:  1  █
  Attaque  3 → dégâts:  5  █████
  Attaque  4 → dégâts:  4  ████
  Attaque  5 → dégâts:  1  █
  Attaque  6 → dégâts:  4  ████
  Attaque  7 → dégâts:  3  ███
  Attaque  8 → dégâts:  6  ██████
  Attaque  9 → dégâts:  2  ██
  Attaque 10 → dégâts:  6  ██████


## 🏆 Feature 3 — Duel 2v2
Deux équipes s'affrontent tour par tour. Chaque vivant attaque un ennemi vivant aléatoire.  
Les morts ne participent plus — garanti par test BDD.

In [4]:
team_a = Team('Équipe A', [
    Character('Alice', endurance=2, level=1, force=1),
    Character('Bob',   endurance=0, level=2, force=2),
])
team_b = Team('Équipe B', [
    Character('Charlie', endurance=3, level=1, force=0),
    Character('Dave',    endurance=1, level=0, force=3),
])

print('=== Avant le duel ===')
for m in team_a.members + team_b.members:
    print(f'  {m.name:10s} HP: {m.health}')

winner = duel(team_a, team_b)

print()
print('=== Après le duel ===')
for m in team_a.members + team_b.members:
    statut = '💀 mort' if m.is_dead() else f'❤️  {m.health} HP restants'
    print(f'  {m.name:10s} {statut}')
print()
print(f'🏆 Vainqueur : {winner.name}')

=== Avant le duel ===
  Alice      HP: 14
  Bob        HP: 14
  Charlie    HP: 15
  Dave       HP: 11

=== Après le duel ===
  Alice      ❤️  7 HP restants
  Bob        💀 mort
  Charlie    💀 mort
  Dave       💀 mort

🏆 Vainqueur : Équipe A


## 🛡️ Feature 4 — Armure selon l'équipement
`armor = sum(e.armor for e in equipment)`  
Les dégâts reçus sont réduits : `dégâts nets = max(0, raw - armor)`

In [5]:
attacker = Character('Attaquant', force=4)
d = 1 + attacker.force + 2 * attacker.level
print(f'Attaquant (FOR=4) → D max: {d}')
print()

configs = [
    ('Aucun',                   []),
    ('Cuir',                    [Equipment('Cuir',    armor=2)]),
    ('Cotte de mailles',        [Equipment('Cotte',   armor=5)]),
    ('Plaques',                 [Equipment('Plaques', armor=8)]),
    ('Cuir + Bouclier',         [Equipment('Cuir', armor=2), Equipment('Bouclier', armor=3)]),
]

print(f'{"Équipement":<22} {"Armure":>6}  {"Dégâts reçus":>12}  {"HP restants":>11}')
print('-' * 58)
for label, eqs in configs:
    target = Character('Cible')
    for eq in eqs:
        target.equipment.append(eq)
    hp_before = target.health
    attacker.attack(target)
    degats = hp_before - target.health
    print(f'  {label:<20} {target.armor:>6}  {degats:>12}  {target.health:>11}')

Attaquant (FOR=4) → D max: 5

Équipement             Armure  Dégâts reçus  HP restants
----------------------------------------------------------
  Aucun                     0             3            7
  Cuir                      2             1            9
  Cotte de mailles          5             0           10
  Plaques                   8             0           10
  Cuir + Bouclier           5             0           10


## 🗡️ Feature 5 — Dégâts selon l'arme
L'arme remplace le `1` de base : `D max = weapon.damage + force + 2 × niveau`  
Sans arme : `weapon = None`, base = `1`.

In [6]:
dagger = Weapon('Dague', damage=1)
sword  = Weapon('Épée',  damage=3)
axe    = Weapon('Hache', damage=5)
spear  = Weapon('Lance', damage=4)

print('Armes disponibles :')
for w in [dagger, sword, axe, spear]:
    print(f'  {w.name:10s} → damage: {w.damage}')

print()
print('Impact sur un combattant (FOR=2, LVL=1) :')
print(f'{"Arme":<12} {"D max":>6}  {"5 jets observés"}')
print('-' * 50)

for weapon in [None, dagger, sword, axe, spear]:
    fighter = Character('Combattant', force=2, level=1)
    fighter.weapon = weapon
    base = fighter.weapon.damage if fighter.weapon else 1
    d = base + fighter.force + 2 * fighter.level
    label = weapon.name if weapon else 'Aucune'

    degats_list = []
    for _ in range(5):
        target = Character('Cible')
        fighter.attack(target)
        degats_list.append(10 - target.health)

    print(f'  {label:<10} {d:>6}  {degats_list}')

Armes disponibles :
  Dague      → damage: 1
  Épée       → damage: 3
  Hache      → damage: 5
  Lance      → damage: 4

Impact sur un combattant (FOR=2, LVL=1) :
Arme          D max  5 jets observés
--------------------------------------------------
  Aucune          5  [3, 1, 0, 2, 4]
  Dague           5  [5, 1, 2, 3, 4]
  Épée            7  [3, 7, 6, 4, 6]
  Hache           9  [8, 9, 4, 0, 0]
  Lance           8  [1, 8, 6, 6, 7]


### 🗡️🛡️ Arme vs Armure — jet maximum (mocké)
On fixe le jet à D max pour montrer clairement l'interaction.

In [7]:
from unittest.mock import patch

combos = [
    ('Aucune',  None,  'Aucune',          []),
    ('Aucune',  None,  'Cuir (2)',         [Equipment('Cuir',    armor=2)]),
    ('Épée',    sword, 'Aucune',           []),
    ('Épée',    sword, 'Cuir (2)',         [Equipment('Cuir',    armor=2)]),
    ('Hache',   axe,   'Plaques (8)',      [Equipment('Plaques', armor=8)]),
]

print(f'{"Arme":<10} {"Armure cible":<14} {"D max":>6}  {"Armure":>6}  {"Dégâts nets":>11}')
print('-' * 55)
for w_label, weapon, e_label, eqs in combos:
    att = Character('Att', force=2, level=1)
    att.weapon = weapon
    base = att.weapon.damage if att.weapon else 1
    d = base + att.force + 2 * att.level

    tgt = Character('Cible')
    for eq in eqs:
        tgt.equipment.append(eq)

    hp_before = tgt.health
    with patch('rpg.character.randint', return_value=d):
        att.attack(tgt)
    degats = hp_before - tgt.health
    print(f'  {w_label:<8} {e_label:<14} {d:>6}  {tgt.armor:>6}  {degats:>11}')

Arme       Armure cible    D max  Armure  Dégâts nets
-------------------------------------------------------
  Aucune   Aucune              5       0            5
  Aucune   Cuir (2)            5       2            3
  Épée     Aucune              7       0            7
  Épée     Cuir (2)            7       2            5
  Hache    Plaques (8)         9       8            1


## 🎬 Scénario complet — Toutes les features en action
Duel 2v2 avec stats, armes et armures combinées.

In [8]:
def d_max(c):
    base = c.weapon.damage if c.weapon else 1
    return base + c.force + 2 * c.level

arthur = Character('Arthur',  endurance=4, level=2, force=3)
arthur.weapon = sword
arthur.equipment.append(Equipment('Cotte', armor=5))

merlin = Character('Merlin',  endurance=1, level=3, force=1)
merlin.weapon = spear

mordred = Character('Mordred', endurance=2, level=2, force=4)
mordred.weapon = axe
mordred.equipment.append(Equipment('Cuir', armor=2))

morgan = Character('Morgan',  endurance=3, level=1, force=2)
morgan.equipment.append(Equipment('Bouclier', armor=3))

team1 = Team('Les Héros',   [arthur, merlin])
team2 = Team('Les Vilains', [mordred, morgan])

print('⚔️  DUEL FINAL ⚔️')
for team in [team1, team2]:
    print(f'\n  🏴 {team.name}')
    for m in team.members:
        w = m.weapon.name if m.weapon else 'Aucune'
        eqs = ', '.join(e.name for e in m.equipment) if m.equipment else 'Aucun'
        print(f'    {m.name:10s}  HP:{m.health:3d}  D max:{d_max(m):2d}  Armure:{m.armor}  Arme:{w}  Équip:{eqs}')

print()
winner = duel(team1, team2)

print('Résultat :')
for team in [team1, team2]:
    for m in team.members:
        statut = '💀 mort' if m.is_dead() else f'❤️  {m.health} HP'
        print(f'  {m.name:10s} {statut}')

print()
print(f'🏆 Victoire : {winner.name} !')

⚔️  DUEL FINAL ⚔️

  🏴 Les Héros
    Arthur      HP: 18  D max:10  Armure:5  Arme:Épée  Équip:Cotte
    Merlin      HP: 17  D max:11  Armure:0  Arme:Lance  Équip:Aucun

  🏴 Les Vilains
    Mordred     HP: 16  D max:13  Armure:2  Arme:Hache  Équip:Cuir
    Morgan      HP: 15  D max: 5  Armure:3  Arme:Aucune  Équip:Bouclier

Résultat :
  Arthur     ❤️  15 HP
  Merlin     💀 mort
  Mordred    💀 mort
  Morgan     💀 mort

🏆 Victoire : Les Héros !
